# ╔══════════════════════════════════════════════════════════════════════════╗
# ║               CREDENTIALS REFERENCE  (for Secret Scope setup)           ║
# ║  Secret Scope name : retail-secrets  |  Key: adls-key                   ║
# ║  adls-key value    : <YOUR_ADLS_STORAGE_KEY>            ║
# ║                      yh3N+vKNzV1a57zCyRf02C5zOn8qHPRyCrL2bW+           ║
# ║                      AStoFWOZg==                                         ║
# ║  See CREDENTIAL_SETUP.md for full setup instructions.                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

In [0]:
storage_account_name = "stretaildatalake122"
storage_key = dbutils.secrets.get(scope="retail-secrets", key="adls-key")
container_name = "bronze"

spark.conf.set(
  f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
  storage_key
)

dbutils.fs.ls(f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/")

[FileInfo(path='wasbs://bronze@stretaildatalake122.blob.core.windows.net/_checkpoints/', name='_checkpoints/', size=0, modificationTime=0),
 FileInfo(path='wasbs://bronze@stretaildatalake122.blob.core.windows.net/sales/', name='sales/', size=0, modificationTime=0),
 FileInfo(path='wasbs://bronze@stretaildatalake122.blob.core.windows.net/sales_stream/', name='sales_stream/', size=0, modificationTime=0),
 FileInfo(path='wasbs://bronze@stretaildatalake122.blob.core.windows.net/sales_stream_raw/', name='sales_stream_raw/', size=0, modificationTime=0)]

In [0]:
storage_account_name = "stretaildatalake122"
storage_key = dbutils.secrets.get(scope="retail-secrets", key="adls-key")
container_name = "silver"

spark.conf.set(
  f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
  storage_key
)

dbutils.fs.ls(f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/")

[]

In [0]:
storage_account_name = "stretaildatalake122"
storage_key = dbutils.secrets.get(scope="retail-secrets", key="adls-key")
container_name = "gold"

spark.conf.set(
  f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
  storage_key
)

dbutils.fs.ls(f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/")

[]

In [0]:
# ── Configuration ──────────────────────────────────────────────
STORAGE_ACCOUNT = "stretaildatalake122"
STORAGE_KEY     = dbutils.secrets.get(scope="retail-secrets", key="adls-key")  

# ── Set key on Spark session (covers all containers at once) ───
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    STORAGE_KEY
)

print("✅ Storage key configured on Spark session")

✅ Storage key configured on Spark session


In [0]:
STORAGE_ACCOUNT = "stretaildatalake122"
STORAGE_KEY     = dbutils.secrets.get(scope="retail-secrets", key="adls-key")

# Ensure key is configured on this session
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    STORAGE_KEY
)

# ── Helper to build full Blob path ─────────────────────────────
def blob_path(container):
    return f"wasbs://{container}@{STORAGE_ACCOUNT}.blob.core.windows.net/"

# ── List each container ────────────────────────────────────────
for container in ["bronze", "silver", "gold"]:
    print(f"\n── {container} ──")
    display(dbutils.fs.ls(blob_path(container)))


── bronze ──


path,name,size,modificationTime
wasbs://bronze@stretaildatalake122.blob.core.windows.net/sales/,sales/,0,0



── silver ──


path,name,size,modificationTime
wasbs://silver@stretaildatalake122.blob.core.windows.net/sales_clean/,sales_clean/,0,0



── gold ──


path,name,size,modificationTime
wasbs://gold@stretaildatalake122.blob.core.windows.net/sales_summary/,sales_summary/,0,0


In [0]:
# ── ONE-TIME SETUP: Create Secret Scope + store ADLS key ───────
# Run this cell ONCE, then never again. Delete it after if you like.
import requests

host  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Step 1: Create the scope (will say RESOURCE_ALREADY_EXISTS if it exists — that's fine)
resp = requests.post(f"{host}/api/2.0/secrets/scopes/create",
                     headers=headers,
                     json={"scope": "retail-secrets"})
print("Create scope:", resp.status_code, resp.text)

# Step 2: Store the ADLS storage key
resp = requests.post(f"{host}/api/2.0/secrets/put",
                     headers=headers,
                     json={"scope": "retail-secrets",
                           "key": "adls-key",
                           "string_value": "<YOUR_ADLS_STORAGE_KEY>"})
print("Put adls-key:", resp.status_code, resp.text)

# Step 3: Verify
try:
    val = dbutils.secrets.get(scope="retail-secrets", key="adls-key")
    print(f"\n✅ Secret scope ready — adls-key length: {len(val)} chars")
except Exception as e:
    print(f"\n❌ Verification failed: {e}")

Create scope: 200 {}
Put adls-key: 200 {}

✅ Secret scope ready — adls-key length: 88 chars
